# PyLauncher Parameter Sweeps with dapi

This notebook demonstrates how to use dapi's parameter sweep utilities to generate
PyLauncher task lists and submit sweep jobs on DesignSafe.

**PyLauncher** runs many independent serial tasks within a single SLURM allocation —
ideal for parameter studies, Monte Carlo simulations, and batch processing.

For an OpenSees-specific example, see [pylauncher_opensees.ipynb](pylauncher_opensees.ipynb).

In [ ]:
%pip install dapi --quiet

In [ ]:
import os
from pathlib import Path
from dapi import DSClient

ds = DSClient()

# On DesignSafe JupyterHub, ~/MyData is /home/jupyter/MyData.
# Locally, we use a local directory but the Tapis path stays /MyData/...
MYDATA = Path(os.environ.get("JUPYTER_SERVER_ROOT", os.path.expanduser("~"))) / "MyData"

## Write the script

A simple script that takes `--alpha`, `--beta`, and `--output` via command line,
computes `result = alpha * beta`, and writes a JSON output file.

In [ ]:
input_dir = MYDATA / "pylauncher_demo"
input_dir.mkdir(parents=True, exist_ok=True)

simulate_script = '''\
"""simulate.py — minimal demo script for PyLauncher parameter sweeps.

Accepts --alpha, --beta, and --output via command line.
Computes result = alpha * beta and writes it to the output directory.
"""
import argparse
import os
import json

parser = argparse.ArgumentParser()
parser.add_argument("--alpha", type=float, required=True)
parser.add_argument("--beta", type=float, required=True)
parser.add_argument("--output", type=str, required=True)
args = parser.parse_args()

os.makedirs(args.output, exist_ok=True)

result = args.alpha * args.beta
summary = {"alpha": args.alpha, "beta": args.beta, "result": result}

outfile = os.path.join(args.output, "result.json")
with open(outfile, "w") as f:
    json.dump(summary, f, indent=2)

print(f"alpha={args.alpha}, beta={args.beta} -> result={result:.4f} written to {outfile}")
'''

(input_dir / "simulate.py").write_text(simulate_script)
print(f"Wrote {input_dir}/simulate.py")

## Define the sweep

In [ ]:
sweep = {
    "ALPHA": [0.3, 0.5, 3.7],
    "BETA": [1.1, 2.0, 3.0],
}

## Preview (dry run)

In [ ]:
ds.jobs.parametric_sweep.generate(
    "python3 simulate.py --alpha ALPHA --beta BETA --output out_ALPHA_BETA",
    sweep,
    preview=True,
)

## Generate sweep files

In [ ]:
commands = ds.jobs.parametric_sweep.generate(
    "python3 simulate.py --alpha ALPHA --beta BETA --output out_ALPHA_BETA",
    sweep,
    str(input_dir),
    debug="host+job",
)

print(f"Generated {len(commands)} task commands\n")
print("=== runsList.txt ===")
print((input_dir / "runsList.txt").read_text())

print("=== call_pylauncher.py ===")
print((input_dir / "call_pylauncher.py").read_text())

print("=== Files in input directory ===")
for fn in sorted(os.listdir(input_dir)):
    print(f"  {fn}")

## Submit

Replace `your_allocation` with your TACC allocation and uncomment to run.

In [ ]:
# job = ds.jobs.parametric_sweep.submit(
#     "/MyData/pylauncher_demo/",
#     app_id="designsafe-agnostic-app",
#     allocation="your_allocation",
#     node_count=1,
#     cores_per_node=48,
#     max_minutes=30,
# )
# job.monitor()